# Ganguly Video Generator (free Colab version)

1. Menu **Runtime**, then **Change runtime type**, then choose **T4 GPU** and Save.
2. Menu **Runtime**, then **Run all**. Wait for the last cells to finish (about 3 minutes).
3. The app appears under the cell named "Open the app". In the app, pick **Fast** quality, about 25 steps, and a short length such as 10 seconds for your first video.
4. The first video downloads the model (several GB), so it takes a long time. Keep this tab open.
5. Before you close Colab, run the last cell to download your newest video. Videos are deleted when the session ends.


In [ ]:
!nvidia-smi -L
!pip -q install -U flask diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg bitsandbytes pillow

In [ ]:
%%writefile ganguly_video_generator.py
#!/usr/bin/env python3
"""Ganguly Video Generator - turn a prompt + attached files into video, locally.

Setup (NVIDIA GPU, Python 3.10+):
  pip install flask torch diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow numpy
Run:
  python ganguly_video_generator.py   ->  open http://127.0.0.1:7860

No quotas anywhere: the queue is unbounded and video length is unbounded. Long videos are
made clip by clip (each clip starts from the previous clip's last frame) and streamed to disk,
so memory stays flat however long the video is. Separate scenes with a blank line.

Attachments: images -> start frame of each scene (in order) | one video -> extended by the new
scenes | audio -> soundtrack (looped to fit) | .txt/.md -> script appended to the prompt.
"""
import os, re, math, queue, random, threading, subprocess, uuid
import numpy as np, imageio, imageio_ffmpeg
from PIL import Image, ImageOps
from flask import Flask, request, jsonify, send_file, abort, Response
from werkzeug.utils import secure_filename

MODEL = os.environ.get("GVG_MODEL", "Lightricks/LTX-Video")
OUT = os.path.abspath("ganguly_videos"); os.makedirs(OUT, exist_ok=True)
FPS, CLIP = 24, 121                      # 121 frames is about 5 s per clip
SIZES = {"fast": (512, 320), "standard": (704, 480), "hd": (768, 512)}
EXT = dict(img={"png", "jpg", "jpeg", "webp", "bmp"}, vid={"mp4", "mov", "mkv", "webm", "avi"},
           aud={"mp3", "wav", "m4a", "aac", "ogg", "flac"}, txt={"txt", "md"})
app, JOBS, Q, _P = Flask(__name__), {}, queue.Queue(), {}


def pipes():
    if not _P:
        import torch
        from diffusers import LTXPipeline, LTXImageToVideoPipeline
        if not torch.cuda.is_available():
            raise RuntimeError("An NVIDIA GPU with CUDA is required for local generation.")
        kw = {}
        if os.environ.get("GVG_LOWMEM"):                  # small GPUs / free Colab: 8-bit text encoder kept on the GPU
            from transformers import T5EncoderModel, BitsAndBytesConfig
            kw["text_encoder"] = T5EncoderModel.from_pretrained(
                MODEL, subfolder="text_encoder", torch_dtype=torch.bfloat16,
                quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": 0})
        t2v = LTXPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16, **kw)
        i2v = LTXImageToVideoPipeline.from_pipe(t2v)
        big = torch.cuda.get_device_properties(0).total_memory > 22e9
        for p in (t2v, i2v):
            p.vae.enable_tiling()
            if kw: p.transformer.to("cuda"); p.vae.to("cuda")
            elif big: p.to("cuda")
            else: p.enable_model_cpu_offload()
        _P.update(t2v=t2v, i2v=i2v)
    return _P


def split_scenes(text):
    parts = [re.sub(r"^\s*scene\s*\d*\s*[:.\-]\s*", "", s, flags=re.I).strip()
             for s in re.split(r"\n\s*\n", text.strip())]
    return [s for s in parts if s]


def plan(n):                              # frames wanted -> list of valid (8k+1) clip lengths
    out = []
    while n > 0:
        c = min(n, CLIP - 1); out.append(math.ceil(c / 8) * 8 + 1); n -= c
    return out


def run(j):
    import torch
    P = pipes(); W, H = SIZES[j["size"]]
    scenes = split_scenes(j["prompt"])
    plans = [plan(max(24, round(j["seconds"] * FPS / len(scenes)))) for _ in scenes]
    total, done = sum(map(len, plans)), 0
    imgs = [ImageOps.fit(Image.open(p).convert("RGB"), (W, H)) for p in j["images"]]
    raw = os.path.join(j["dir"], "raw.mp4")
    w = imageio.get_writer(raw, fps=FPS, macro_block_size=1)
    last = None
    if j["video"]:                        # extend an attached video
        r = imageio.get_reader(j["video"])
        for fr in r:
            last = ImageOps.fit(Image.fromarray(fr).convert("RGB"), (W, H)); w.append_data(np.asarray(last))
        r.close()
    stop = False
    for si, (sc, pl) in enumerate(zip(scenes, plans)):
        start = imgs[si] if si < len(imgs) else last
        for ci, nf in enumerate(pl):
            if j["cancel"]: stop = True; break
            j["msg"] = f"scene {si + 1} of {len(scenes)}, clip {ci + 1} of {len(pl)}"
            kw = dict(prompt=sc, negative_prompt=j["negative"], width=W, height=H, num_frames=nf,
                      num_inference_steps=j["steps"], generator=torch.Generator("cpu").manual_seed(j["seed"] + done))
            fr = (P["i2v"](image=start, **kw) if start is not None else P["t2v"](**kw)).frames[0]
            for im in (fr[1:] if (start is not None and start is last) else fr):
                w.append_data(np.asarray(im))
            last = start = fr[-1]; done += 1; j["progress"] = done / total
            torch.cuda.empty_cache()
        if stop: break
    w.close()
    final = os.path.join(OUT, j["id"] + ".mp4")
    if j["audio"]:
        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", raw, "-stream_loop", "-1", "-i", j["audio"],
                        "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-shortest", final],
                       check=True, capture_output=True)
    else:
        os.replace(raw, final)
    j["status"], j["msg"], j["progress"] = "done", "stopped early" if stop else "", 1.0 if not stop else j["progress"]


def worker():
    while True:
        j = Q.get()
        try:
            j["status"] = "running"; run(j)
        except Exception as e:
            j["status"], j["msg"] = "error", str(e)[:300]


@app.post("/generate")
def generate():
    f, jid = request.form, uuid.uuid4().hex[:8]
    d = os.path.join(OUT, "work_" + jid); os.makedirs(d)
    j = dict(id=jid, dir=d, prompt=f.get("prompt", "").strip(), negative=f.get("negative", "").strip() or
             "worst quality, blurry, distorted, jittery, watermark, text", size=f.get("size", "standard"),
             seconds=max(1.0, float(f.get("seconds") or 5)), steps=int(f.get("steps") or 40),
             seed=int(f.get("seed") or random.randint(0, 2**31)), images=[], video=None, audio=None,
             status="queued", progress=0.0, msg="", cancel=False)
    for u in request.files.getlist("files"):
        n = secure_filename(u.filename or "")
        if not n: continue
        p = os.path.join(d, n); u.save(p); e = n.rsplit(".", 1)[-1].lower()
        if e in EXT["img"]: j["images"].append(p)
        elif e in EXT["vid"] and not j["video"]: j["video"] = p
        elif e in EXT["aud"] and not j["audio"]: j["audio"] = p
        elif e in EXT["txt"]: j["prompt"] += "\n\n" + open(p, errors="ignore").read()
    if not j["prompt"].strip():
        if not (j["images"] or j["video"]): return Response("Write a prompt or attach a file.", 400)
        j["prompt"] = "Smooth, natural, cinematic motion continuing this scene"
    if j["size"] not in SIZES: j["size"] = "standard"
    JOBS[jid] = j; Q.put(j)
    return jsonify(id=jid)


@app.get("/jobs")
def jobs():
    return jsonify([{k: j[k] for k in ("id", "prompt", "status", "progress", "msg")} for j in JOBS.values()])


@app.post("/cancel/<jid>")
def cancel(jid):
    if jid in JOBS: JOBS[jid]["cancel"] = True
    return "", 204


@app.get("/video/<jid>")
def video(jid):
    p = os.path.join(OUT, jid + ".mp4")
    if not re.fullmatch(r"[0-9a-f]{8}", jid) or not os.path.exists(p): abort(404)
    return send_file(p, mimetype="video/mp4", conditional=True)


@app.get("/")
def home(): return Response(HTML, mimetype="text/html")


HTML = r"""<!doctype html><html lang=en><meta charset=utf-8><meta name=viewport content="width=device-width,initial-scale=1">
<title>Ganguly Video Generator</title>
<style>
:root{--bg:#e8ecef;--ink:#0f1b26;--mut:#5a6874;--blue:#2b4a7a;--line:#c5cdd4;--amber:#d99a00}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.5 system-ui,sans-serif}
header{background:var(--ink);color:#fff;padding:18px 28px;border-bottom:6px solid var(--amber)}
h1{margin:0;font:700 clamp(1.5rem,3vw,2.2rem)/1.1 Georgia,serif;letter-spacing:.01em}
header p{margin:4px 0 0;color:#aab6c1;font-size:.95rem}
main{display:grid;grid-template-columns:minmax(300px,420px) 1fr;gap:28px;padding:28px;max-width:1200px;margin:auto}
@media(max-width:800px){main{grid-template-columns:1fr;padding:16px}}
form{display:grid;gap:12px;align-content:start}label{display:grid;gap:4px;font-weight:600;font-size:.9rem}
textarea,input,select{font:inherit;padding:9px 10px;border:1px solid var(--line);background:#fff;border-radius:3px;color:var(--ink)}
textarea{min-height:150px;resize:vertical}.row{display:grid;grid-template-columns:1fr 1fr;gap:12px}
small{color:var(--mut);font-weight:400}
button{font:600 1rem system-ui;padding:11px 16px;border:0;border-radius:3px;background:var(--blue);color:#fff;cursor:pointer}
button:disabled{opacity:.5}button.s{background:none;color:var(--mut);padding:4px 0;text-decoration:underline;font-size:.85rem}
:focus-visible{outline:3px solid var(--amber);outline-offset:2px}
#jobs{display:grid;gap:16px;align-content:start}.job{border-left:6px solid var(--blue);padding:4px 0 4px 14px}
.job b{display:block;font-weight:600}.job p{margin:2px 0 6px;color:var(--mut);font-size:.9rem}
progress{width:100%;height:8px;accent-color:var(--amber)}video{width:100%;max-width:640px;background:#000;display:block;margin:6px 0}
.empty{color:var(--mut);border:2px dashed var(--line);padding:28px}
</style>
<header><h1>Ganguly Video Generator</h1><p>Describe it, attach anything that helps, and make videos of any length with no limit on how many.</p></header>
<main>
<form id=f>
<label>Prompt <small>Blank line = new scene. Each scene gets the full duration share.</small>
<textarea name=prompt placeholder="A fishing boat leaves a Kolkata ghat at dawn, mist on the Hooghly, warm light on the water."></textarea></label>
<label>Attachments <small>Images, a video to extend, audio, or a .txt script</small>
<input type=file name=files multiple accept="image/*,video/*,audio/*,.txt,.md"></label>
<div class=row><label>Length (seconds)<input type=number name=seconds value=10 min=1 step=any></label>
<label>Quality<select name=size><option value=fast>Fast, 512x320<option value=standard selected>Standard, 704x480<option value=hd>Sharper, 768x512</select></label></div>
<div class=row><label>Steps<input type=number name=steps value=40 min=10></label><label>Seed <small>blank = random</small><input type=number name=seed></label></div>
<label>Avoid <input name=negative placeholder="blurry, distorted, text"></label>
<button id=go>Make video</button></form>
<section id=jobs><div class=empty id=none>Your videos appear here. Start as many as you like; they run in order.</div></section>
</main>
<script>
const $=s=>document.querySelector(s);
$('#f').onsubmit=async e=>{e.preventDefault();const b=$('#go');b.disabled=true;
 const r=await fetch('/generate',{method:'POST',body:new FormData(e.target)});
 if(!r.ok)alert(await r.text());b.disabled=false;poll()};
async function poll(){
 for(const j of await (await fetch('/jobs')).json()){
  $('#none')?.remove();
  let c=document.getElementById('j'+j.id);
  if(!c){c=document.createElement('div');c.id='j'+j.id;c.className='job';
   c.innerHTML='<b></b><p></p><progress max=1></progress><div></div><button type=button class=s>Stop and keep what is made</button>';
   c.querySelector('b').textContent=j.prompt.slice(0,100);
   c.querySelector('button').onclick=()=>fetch('/cancel/'+j.id,{method:'POST'});$('#jobs').prepend(c)}
  c.querySelector('p').textContent=j.status+(j.msg?', '+j.msg:'');c.querySelector('progress').value=j.progress;
  if(j.status=='done'&&!c.querySelector('video')){c.children[3].innerHTML=`<video controls src=/video/${j.id}></video><a download href=/video/${j.id}>Download</a>`}
  if(j.status=='done'||j.status=='error')c.querySelector('button')?.remove()}}
setInterval(poll,2000);poll();
</script></html>"""

if __name__ == "__main__":
    threading.Thread(target=worker, daemon=True).start()
    print("Ganguly Video Generator: http://127.0.0.1:7860   (first run downloads the model)")
    app.run(host="127.0.0.1", port=7860, threaded=True)


In [ ]:
#@title Open the app
import os, subprocess, time
os.environ["GVG_LOWMEM"] = "1"
log = open("server.log", "w")
server = subprocess.Popen(["python", "ganguly_video_generator.py"], stdout=log, stderr=log)
time.sleep(8)
from google.colab import output
output.serve_kernel_port_as_iframe(7860, height=1000)

In [ ]:
#@title If something goes wrong, run this and send me the text
!tail -n 40 server.log

In [ ]:
#@title Download my newest video
import glob, os
from google.colab import files
v = sorted(glob.glob("ganguly_videos/*.mp4"), key=os.path.getmtime)
files.download(v[-1]) if v else print("No video yet")